In [1]:
import sqlite3
import pandas as pd
import os
import re
import time
from tqdm import tqdm

### Variables from conf file

In [2]:
# database file path
DB_FILE = "../../../drive_data/v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310.db"
NER_TIMEX_DB = "../../../drive_data/v33_ner_timex_20250922-094340.db"

# transaction table to update with status
TRANSACTION_TABLE = "transaction_row"


### Connect to databases

In [3]:
conn_source = sqlite3.connect(NER_TIMEX_DB)  # Source database
conn_target = sqlite3.connect(DB_FILE)  # Target database

cursor_source = conn_source.cursor()
cursor_target = conn_target.cursor()

### Add ner and timex tags

In [4]:
def column_exists(cursor, table, column):
    cursor.execute(f"PRAGMA table_info({table})")
    return any(row[1] == column for row in cursor.fetchall())

In [5]:
def ner_timex_to_table(input_table, cursor1, cursor2, conn1, conn2):
    # Step 1: Retrieve data from database1
    cursor1.execute("SELECT sentence_id, ner_tag, loc FROM ner")
    ners = cursor1.fetchall()  # List of (sentence_id, ner_tag, loc)

    # Step 2: add new column to database2 table
    if not column_exists(cursor2, input_table, "ner_tag"):
        cursor2.execute("ALTER TABLE " + input_table +  " ADD COLUMN ner_tag TEXT")

    # Step 3: Create a temporary table
    cursor2.execute("CREATE TEMP TABLE temp_ner (sentence_id INT, ner_tag TEXT, loc INT)")

    # Step 4: Insert all values into the temp table
    cursor2.executemany("INSERT INTO temp_ner (sentence_id, ner_tag, loc) VALUES (?, ?, ?)", ners)

    cursor2.execute(f"""
        UPDATE {input_table}
        SET ner_tag = temp_ner.ner_tag
        FROM temp_ner
        WHERE {input_table}.sentence_id = temp_ner.sentence_id and {input_table}.loc=temp_ner.loc;
    """)
    
    conn2.commit()
    
    
    
    # Step 1: Retrieve data from database1
    cursor1.execute("SELECT sentence_id, timex_type, loc FROM timex")
    timexes = cursor1.fetchall()  # List of (sentence_id, timex_type, loc)

    # Step 2: add new column to database2 table
    if not column_exists(cursor2, input_table, "timex_tag"):
        cursor2.execute("ALTER TABLE " + input_table +  " ADD COLUMN timex_tag TEXT")

    # Step 3: Create a temporary table
    cursor2.execute("CREATE TEMP TABLE temp_timex (sentence_id INT, timex_type TEXT, loc INT)")

    # Step 4: Insert all values into the temp table
    cursor2.executemany("INSERT INTO temp_timex (sentence_id, timex_type, loc) VALUES (?, ?, ?)", timexes)

    # Step 3: Perform a fast join-based update
    cursor2.execute(f"""
        UPDATE {input_table}
        SET timex_tag = temp_timex.timex_type
        FROM temp_timex
        WHERE {input_table}.sentence_id = temp_timex.sentence_id and {input_table}.loc=temp_timex.loc;
    """)
    
    conn2.commit()

    conn1.close()
    conn2.close()

In [6]:
start = time.time()

ner_timex_to_table(TRANSACTION_TABLE, cursor_source, cursor_target, conn_source, conn_target)

end = time.time()
elapsed_time = end - start
minutes = int(elapsed_time // 60)
seconds = int(elapsed_time % 60)
print(f"time: {minutes} minute(s) and {seconds} second(s)")

time: 3 minute(s) and 11 second(s)


In [7]:
conn_source.close()
conn_target.close()

### Check the results (not part of final workflow)

### Connect to database

In [8]:
# connecting with database
conn = sqlite3.connect(DB_FILE)
cur = conn.cursor()

In [9]:
query = f"SELECT * FROM transaction_row limit 10"

res = pd.read_sql(query, conn)
res

,id,head_id,loc,loc_rel,deprel,form,lemma,feats,parent_loc,pos,sentence_id,status,ekilex_tag,ner_tag,timex_tag
0,1,2,3,-1,obl,lõpus,lõpp,"com,in,sg",None,S,3,,None,None,DATE
1,2,2,5,1,nsubj,Türi,Türi,"gen,prop,sg",None,S,3,syntax-morph conflict,None,LOC,None
2,3,2,6,2,obl,1.,1.,"<?>,ord,roman",None,N,3,,None,None,None
3,4,3,1,-3,obj,Bändi,bänd,"adit,com,sg",None,S,5,syntax-morph conflict,None,None,None
4,5,3,9,-2,nsubj,kidramees,kidramees,"com,nom,sg",None,S,5,,None,None,None
5,6,3,10,-1,aux,ei,ei,"aux,neg",None,V,5,,None,None,None
6,7,3,12,1,obl,keeltele,keel,"all,com,pl",None,S,5,,None,None,None
7,8,3,13,2,compound:prt,pihta,pihta,,None,D,5,,None,None,None
8,9,4,4,-2,nsubj,solist,solist,"com,nom,sg",None,S,5,,None,None,None
9,10,4,5,-1,aux,ei,ei,"aux,neg",None,V,5,,None,None,None


In [11]:
query = """
SELECT 
    ekilex_tag,
    ner_tag,
    CASE 
        WHEN timex_tag IS NOT NULL THEN 'TIME'
        ELSE NULL
    END AS timex
FROM transaction_row
GROUP BY 
    ekilex_tag,
    ner_tag,
    timex
"""

res = pd.read_sql(query, conn)
res

,ekilex_tag,ner_tag,timex
0,None,None,None
1,None,None,TIME
2,None,LOC,None
3,None,LOC,TIME
4,None,ORG,None
5,None,ORG,TIME
6,None,PER,None
7,None,PER,TIME
8,alive,None,None
9,alive,None,TIME


In [12]:
#res.to_csv("tags_combinations.csv", index=False, sep=",", encoding="utf-8")

In [13]:
conn.close()